# From a plain-English strategy to a paper portfolio

This bot lets you describe a trading idea in plain English. It turns the idea into a short, checked
strategy definition, tests it on past S&P 500 data with trading costs, and compares it with simple
benchmarks. Approved strategies are then paper traded. No real money is used.

In this notebook you will see the idea, one example strategy, the rules that keep the test honest,
and the results.

**The numbers below are SIMULATED DATA.** The notebook runs on a made-up data feed (`EODHD_MODE=fake`),
so nothing printed here is a finding. It shows how the tool works, not whether a strategy makes money.

**Which stocks.** Live runs use a frozen random sample of the historical S&P 500 members, not the
whole index. The first code cell prints the exact sentence, with a short note on what a sample
can and cannot show. The fake feed has its own small made-up universe, so the simulated cells run
on all of it.

In [ ]:
import os
import sys
from pathlib import Path

# Fake mode must be set before any bot module reads the data mode.
os.environ["EODHD_MODE"] = "fake"
# The fake feed uses made-up SIM tickers that are not in the real frozen sample, so the
# simulated cells run on the whole fake universe. Live runs use the sample.
os.environ["EODHD_UNIVERSE"] = "full"

# Work from the repo root whether the notebook starts there or in notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "bot").is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from datetime import date

import pandas as pd

from bot.config import SIMULATED_LABEL, data_mode
from bot.universe import SAMPLE_LIMITATIONS, universe_label

BANNER = f"*** {SIMULATED_LABEL} (data mode: {data_mode()}). Not a finding. ***"
print(BANNER)
print("Universe in these simulated cells:", universe_label())
print("Universe in the live results:", universe_label("sample"))
print("Limitations of the live sample:", SAMPLE_LIMITATIONS)

## 1. The idea

You write: "Buy stocks that open 3% or more below yesterday's close when there is no news. Sell
after 3 days." The agent does not write code. It fills in a small form (a **strategy definition**)
and Python checks every field. Then Python does all the maths. The agent only explains the results.

The rule of thumb: the agent talks, Python counts.

## 2. The strategy definition

Here is that idea as a definition. It is plain JSON. `describe_rules` turns it back into English so
you can check that the bot understood you.

In [ ]:
from bot.strategy.schema import backtest_windows, describe_rules, strategy_from_json

GAP_DOWN_JSON = """
{
  "name": "Gap down, no news",
  "description_plain_english": "Buy stocks that gap down 3%+ on no news. Sell after 3 days.",
  "version": 1,
  "entry": [
    {"block": "gap", "direction": "down", "min_gap_pct": 3},
    {"block": "no_news", "lookback_sessions": 1}
  ],
  "exit": {"holding_days": 3},
  "sizing": {"max_positions": 10}
}
"""
strategy = strategy_from_json(GAP_DOWN_JSON)
print(describe_rules(strategy))

## 3. Who was in the S&P 500 on that day

The index changes over time. If we test on today's members, we only buy stocks that survived. That
flatters the result. This is called **survivorship bias**.

So the bot keeps a **membership history**. On any past day it may only consider stocks that were in
the index that day. The cell below picks a stock that left the index and shows True (member) or False
(not a member) around its last day.

In [ ]:
from bot.data import eodhd_client as client
from bot.data.membership import build_membership_history, membership_matrix

history = build_membership_history(client.get_index_history())
left = [s for s in history.spells if s.end is not None and s.end.year >= 2017]
spell = sorted(left, key=lambda s: s.end)[0]
days = pd.bdate_range(spell.end - pd.Timedelta(days=4), spell.end + pd.Timedelta(days=4))
print(BANNER)
print(f"{spell.symbol} was in the index until {spell.end}:")
print(membership_matrix(history, days)[spell.symbol].to_string())

## 4. What a decision may see

Each decision is made after the 16:00 New York close on day D. The bot may use only data that was
known by then. Nothing from day D+1 or later gets in.

| Data | What the decision can use |
| --- | --- |
| Prices | Rows up to and including day D |
| News | Articles published before 16:00 on day D |
| Index members | The list as of day D |
| Orders | Placed on day D, filled at the next open (D+1) |

The cell below loads the market and cuts it at one decision day. The last row the bot can see is
that day, never later. In fake mode the data is built from a simulated feed.

In [ ]:
import time

from bot.config import BACKTEST_END
from bot.data.market import PRICE_START, load_market_report
from bot.strategy.blocks import slice_to_decision_date

started = time.time()
market, report = load_market_report(PRICE_START, BACKTEST_END)
print(BANNER)
print(f"loaded {report.universe_size} symbols, {len(market.trading_days)} trading days")

DECISION_DAY = date(2022, 2, 10)
data = slice_to_decision_date(market, DECISION_DAY)
print("decision day:", DECISION_DAY, "| last row visible:", data.market.trading_days[-1].date())
print("index members that day:", len(data.members))

### Finding: news coverage switches on in 2021

The news data does not thin out gradually. In the live sample, the typical stock had about 8 articles
in 2019 and about 42 in 2020, then about 1,500 in 2021: a jump of roughly 35 times. Before 2021, "no
news" mostly means "nothing was recorded", not "a quiet stock". So news strategies are tested on
**2021 to 2023 only**. 2019 and 2020 appear only as a coverage footnote, never as results. The
cell below shows the same table for whatever data is loaded. In fake mode the curve is made up and
differs from the live one.

In [ ]:
# Articles per member ticker-year: median and p10 across stocks that were in the index that year.
years = market.article_count.index.year
articles = market.article_count.groupby(years).sum(min_count=1)
was_member = market.membership.groupby(market.membership.index.year).any()
rows = []
for year in range(2019, 2024):
    if year not in articles.index:
        continue
    counts = articles.loc[year][was_member.loc[year]].fillna(0)
    rows.append({"year": year, "median": counts.median(), "p10": counts.quantile(0.10)})
coverage = pd.DataFrame(rows).set_index("year")
print(BANNER)
print(coverage.round(1))
if 2020 in coverage.index and 2021 in coverage.index and coverage.loc[2020, "median"] > 0:
    jump = coverage.loc[2021, "median"] / coverage.loc[2020, "median"]
    print(f"median jump 2021 vs 2020: {jump:.1f} times")
coverage["median"].plot.bar(title="Median articles per stock per year (" + data_mode() + " data)");

### Which articles count as company news

An article counts as news about a stock only if it is tagged on at most 2 US symbols. Market
roundups tagged on dozens of tickers are not news about any one company. The rule was chosen on
coverage and article quality before any returns were seen. The results summary also runs the
gap-down strategy under the 1-symbol and 3-symbol rules, as a secondary robustness check.

> **Known EODHD limitation.** Some articles about TerrAscend (Canadian ticker TER, a cannabis
> company) are tagged `TER.US`, which is Teradyne. A symbol-count rule does not remove this
> mis-tag, because those articles are tagged on only one or two symbols. Treat Teradyne news
> counts with care.

## 5. The building blocks

A **building block** is a small test on one idea. It takes the data up to the decision day and
says True or False for each stock. A strategy is a few blocks combined with "and".

| Block | Question it answers |
| --- | --- |
| `gap` | Did the stock open well above or below yesterday's close? |
| `n_day_return` | How much did it move over the last N days? |
| `price_vs_moving_average` | Is it above or below its recent average price? |
| `rsi` | Is it overbought or oversold? (RSI is a 0 to 100 momentum score.) |
| `relative_volume` | Is trading volume unusually high or low? |
| `no_news` / `news_sentiment` | Was there news, and was it positive or negative? |
| `days_to_earnings` / `days_since_earnings` | How close is an earnings report? |
| `spy_filter` | Is the whole market above its long-term average? |

Here are two blocks measured on the decision day, for the five biggest gap-downs.

In [ ]:
from bot.strategy.blocks import measure_gap_pct, measure_news_count

table = pd.DataFrame({"gap_pct": measure_gap_pct(data), "news_count": measure_news_count(data, 1)})
print(BANNER)
print(table.sort_values("gap_pct").head(5).round(2))

## 6. The backtest

A **backtest** replays the strategy over past days. The rules are fixed:

- A signal on day D becomes an order that fills at the **next open**. We never trade at a price we
  could not have known.
- Every fill pays a cost of 5 basis points (0.05%) of the trade value, on each side.
- No leverage. A buy is shrunk so that value plus cost never exceeds cash.
- Splits and dividends are applied to positions that are held.

Because this strategy uses news, it is tested on 2021 to 2023 only. News coverage is too thin before 2021 (see the finding in section 4).

In [ ]:
from bot.backtest.costs import make_cost_model
from bot.backtest.engine import run_backtest
from bot.backtest.metrics import compute_metrics

window = backtest_windows(strategy)[0]
result = run_backtest(strategy, market, window, make_cost_model(5.0))
metrics = compute_metrics(result, market, versions_tried=1)
print(BANNER)
print("window:", window.label, window.start, "to", window.end)
too_few = " (too few trades to judge: under 30)" if metrics.too_few_trades else ""
print(f"closed trades: {metrics.closed_trades}{too_few}")
print("universe:", universe_label())

## 7. Results against SPY and equal-weight

Two yardsticks:

- **SPY**: buy the S&P 500 fund and hold.
- **Equal-weight, same sample**: the average member's daily return, over the same stocks the
  strategy could buy. In the live sample this is the fair comparison. It ignores rebalancing costs,
  so it is slightly flattering. The results summary also shows equal weight on all 741 historical
  members when their prices are downloaded, and says "skipped" when they are not.

Columns: total return, maximum drawdown (the worst fall from a peak) and Sharpe (return per unit of
ups and downs). A strategy with fewer than 30 closed trades is flagged next to its row as too few to
judge. A smaller sample means fewer trades and wider uncertainty.

In [ ]:
def summary_row(label, stats):
    """One table row from a ReturnStats object."""
    return {
        "": label,
        "total_return_pct": stats.total_return_pct,
        "max_drawdown_pct": stats.max_drawdown_pct,
        "sharpe": stats.sharpe,
    }


from bot.reporting import benchmark_labels

flag = " (too few trades: under 30)" if metrics.too_few_trades else ""
print(BANNER)
print(universe_label())
print(
    pd.DataFrame(
        [
            summary_row("Strategy" + flag, metrics.strategy_returns),
            summary_row("SPY", metrics.spy),
            summary_row(benchmark_labels()["equal_weight_sp500"], metrics.equal_weight),
        ]
    )
    .round(2)
    .to_string(index=False)
)

In [ ]:
from bot.backtest.charts import plot_equity_vs_benchmarks
from bot.backtest.metrics import benchmark_curves
from bot.config import output_dir

chart = plot_equity_vs_benchmarks(
    benchmark_curves(result, market), metrics, output_dir() / "walkthrough_equity.png"
)
print("chart saved to", chart)

### Cost sensitivity

The same signals at 0, 5 and 20 basis points per side. A strategy that only works at 0 does not
work.

In [ ]:
from bot.backtest.engine import run_cost_sensitivity

rows = []
for bps, run in run_cost_sensitivity(strategy, market, window).items():
    stats = compute_metrics(run, market, versions_tried=1).strategy_returns
    rows.append(summary_row(f"{bps} bps", stats))
print(BANNER)
print(universe_label())
print(pd.DataFrame(rows).round(2).to_string(index=False))

## 8. Paper trading

Paper trading means trading with pretend money. After each close, the bot runs the same daily
step that the backtest uses, then saves the portfolio. Using the same step means the live path and
the test path cannot drift apart.

## 9. What the agent eval measures

The agent has five jobs: translate a request into a definition, ask when the request is unclear,
confirm before running, decline what is not supported, and explain results without inventing
numbers. `evals/run_evals.py` sends a fixed set of requests to the model and scores each reply.

It calls an outside API, so this notebook only reads the latest saved summary.

In [ ]:
summaries = sorted((ROOT / "outputs" / "simulated" / "evals").glob("*/summary.md"))
if summaries:
    print("latest saved eval summary:", summaries[-1].parent.name)
    print("\n".join(summaries[-1].read_text().splitlines()[:25]))
else:
    print("No saved eval run. Run: uv run python evals/run_evals.py --mode direct")

## Safety

**The sealed holdout.** Data from 2024-01-01 onward is locked away. We never tune or look at it
while building a strategy. Every data call checks the end date and raises an error if it reaches
into the locked period.

**One-time approval.** To unlock it once, the user types `approve`. The bot sends a 6-digit code to
the allowed Telegram user. Only the typed code writes the approval. The holdout then runs a single
time, and paper trading starts.

**The holdout session.** Even an approved request is not enough on its own. The code must also be
inside a `holdout_session`, which opens only when an unused approval exists. The cell below tries
three ways in without approval. All three should fail.

**Caching.** Raw data from EODHD is saved on disk, so repeated runs are fast and free. Saved files
are never committed to git, and they can always be rebuilt.

In [ ]:
from bot.data.holdout import holdout_session

attempts = {
    "load data into 2024": lambda: load_market_report(PRICE_START, date(2024, 1, 31)),
    "approved=True, no session": lambda: load_market_report(
        PRICE_START, date(2024, 1, 31), approved=True
    ),
    "open a session, no approval": lambda: holdout_session("strat_0000000000000000").__enter__(),
}
for name, attempt in attempts.items():
    try:
        attempt()
        print(name, "-> NOT BLOCKED (this would be a bug)")
    except Exception as error:  # the type is printed on purpose
        print(f"{name} -> blocked ({type(error).__name__})")

## For developers

The sections above leave out the internals. They are covered in `docs/design/`:

- `docs/design/data_layer.md`: how prices are rebuilt from raw rows and splits (the adjusted close
  from EODHD already knows the future, so it is not used), the news window across weekends and
  daylight saving changes, and the on-disk cache layers.
- `docs/design/strategy_schema.md`: every field of the strategy definition and its limits.
- `docs/design/backtest.md`: the daily step (splits, fills, stops, valuation, new orders), the
  metrics, and the parity test that replays the paper path against `run_backtest`.
- `tests/test_lookahead.py`: rebuilds the world from only the rows known at many past dates and
  checks that every block gives the same answer as the full-history version.

Run the lookahead test with `uv run pytest tests/test_lookahead.py -q`.